# Graph-pKa (graph neural network) on our counter-ions, with metrics

**Paper:** Xiong, J. *et al.* (2022). *Multi-instance learning of graph neural networks for aqueous pKa prediction.* Bioinformatics 38(3), 792–798. Official code: https://github.com/myzhengSIMM/Graph-pKa

**What is reproduced:** the architecture (AttentiveFP-style message passing, one pKa per atom, multi-instance pooling), the ionizable-site rules, separate acidic and basic models, MSE loss, 70/15/15 split, MC-dropout uncertainty and the applicability domain (max Tanimoto > 0.5).

**What is not:** the authors' training set (S-pKa, 16,595 molecules) and trained weights are not public. The model is **retrained on the Baltruschat & Czodrowski (2020) dataset** (about 5,800 molecules, pKa 2 to 12). Report it as "the Graph-pKa architecture retrained on the Baltruschat & Czodrowski data".

**Metrics reported, the same way as for the random forest:**
1. Held-out 15% of the training data (internal test).
2. The two external test sets shipped with the Baltruschat repo (Novartis, AvLiLuMoVe), never used in training.
3. Our 153 ions that have a measured pKa.

The paper itself reports MAE about 0.55 and R² about 0.92 on its own test set.

Upload `IL_ion_dataset_579_v5_pKa_fixed.csv` when asked. Use a GPU runtime in Colab if you can (CPU also works, about 20 to 30 minutes).

In [1]:
import sys, subprocess, os
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rdkit"], check=True)

## 1. Setup

In [2]:
import math, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, PandasTools
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog('rdApp.*')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE.type == 'cpu':
    print('WARNING: no GPU -- training will be slow. Switch runtime to GPU if you can.')
MAX_EPOCHS = int(os.environ.get("MAX_EPOCHS", 150)); PATIENCE = 20

torch 2.14.1+cu130 | device: cpu


## 2. Training data and the two external test sets (Baltruschat & Czodrowski repository, CC BY 4.0)

In [3]:
REPO = "Machine-learning-meets-pKa"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/czodrowskilab/Machine-learning-meets-pKa.git", REPO], check=True)
train_raw = PandasTools.LoadSDF(f"{REPO}/datasets/combined_training_datasets_unique_no_oe.sdf")
train_raw["pKa"] = train_raw["pKa"].astype(float)
ext = {}
for name, f in [("Novartis", "novartis_cleaned_mono_unique_notraindata.sdf"),
                ("AvLiLuMoVe", "AvLiLuMoVe_cleaned_mono_unique_notraindata.sdf")]:
    e = PandasTools.LoadSDF(f"{REPO}/datasets/{f}"); e["pKa"] = e["pKa"].astype(float); ext[name] = e
print("training molecules:", len(train_raw), train_raw.marvin_pKa_type.value_counts().to_dict())
for k, e in ext.items(): print(k, len(e), e.marvin_pKa_type.value_counts().to_dict())

training molecules: 5916 {'basic': 3262, 'acidic': 2654}
Novartis 280 {'basic': 168, 'acidic': 112}
AvLiLuMoVe 123 {'basic': 97, 'acidic': 26}


## 3. Ionizable sites (paper Section 2.1) and neutralization

In [4]:
uncharger = rdMolStandardize.Uncharger()

def neutralize(mol):
    return uncharger.uncharge(Chem.Mol(mol))

def acidic_sites(mol):
    return [a.GetSymbol() != 'C' and a.GetTotalNumHs() >= 1 for a in mol.GetAtoms()]

def basic_sites(mol):
    return [a.GetSymbol() == 'N' and a.GetFormalCharge() <= 0 for a in mol.GetAtoms()]

# quick check on two real cases
for smi in ['CC(C)Cc1ccc(C(C)C(=O)[O-])cc1', 'Cn1cc[nH+]c1']:
    n = neutralize(Chem.MolFromSmiles(smi))
    print(smi, '->', Chem.MolToSmiles(n),
          '| acidic sites:', sum(acidic_sites(n)), '| basic sites:', sum(basic_sites(n)))

CC(C)Cc1ccc(C(C)C(=O)[O-])cc1 -> CC(C)Cc1ccc(C(C)C(=O)O)cc1 | acidic sites: 1 | basic sites: 0
Cn1cc[nH+]c1 -> Cn1ccnc1 | acidic sites: 0 | basic sites: 2


## 4. Atom and bond features, graphs

In [5]:
ELEMENTS = ['B','C','N','O','F','Si','P','S','Cl','As','Se','Br','Te','I','At']
HYBRID = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
          Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
          Chem.rdchem.HybridizationType.SP3D2]
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
         Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
STEREO = [Chem.rdchem.BondStereo.STEREONONE, Chem.rdchem.BondStereo.STEREOANY,
          Chem.rdchem.BondStereo.STEREOZ, Chem.rdchem.BondStereo.STEREOE]

def one_hot(x, choices, other=True):
    v = [float(x == c) for c in choices]
    if other: v.append(float(x not in choices))
    return v

def atom_features(a):
    return (one_hot(a.GetSymbol(), ELEMENTS)            # element (16)
            + one_hot(a.GetDegree(), [0,1,2,3,4,5])     # degree (7)
            + [float(a.GetFormalCharge())]              # formal charge (1)
            + [float(a.GetNumRadicalElectrons())]       # radical electrons (1)
            + one_hot(a.GetHybridization(), HYBRID)     # hybridization (6)
            + [float(a.GetIsAromatic())]                # aromatic (1)
            + one_hot(a.GetTotalNumHs(), [0,1,2,3,4])   # attached H (6)
            + [float(a.IsInRing())])                    # in ring (1)

def bond_features(b):
    return (one_hot(b.GetBondType(), BONDS, other=False)   # bond type (4)
            + [float(b.GetIsConjugated())]                 # conjugated (1)
            + [float(b.IsInRing())]                        # in ring (1)
            + one_hot(b.GetStereo(), STEREO))              # stereo (5)

_m = Chem.MolFromSmiles('CC(=O)O')
N_ATOM_FEATS = len(atom_features(_m.GetAtomWithIdx(0)))
N_BOND_FEATS = len(bond_features(_m.GetBondWithIdx(0)))
print('atom features:', N_ATOM_FEATS, '| bond features:', N_BOND_FEATS)

atom features: 39 | bond features: 11


In [6]:
def mol_to_graph(mol, kind):
    """Graph dict for one (already neutral) molecule. kind = 'acidic' or 'basic'."""
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    src, dst, ef = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = bond_features(b)
        src += [i, j]; dst += [j, i]; ef += [f, f]      # both directions
    edge_index = torch.tensor([src, dst], dtype=torch.long) if src else torch.zeros((2,0), dtype=torch.long)
    edge_attr = torch.tensor(ef, dtype=torch.float32) if ef else torch.zeros((0, N_BOND_FEATS))
    sites = acidic_sites(mol) if kind == 'acidic' else basic_sites(mol)
    return {'x': x, 'edge_index': edge_index, 'edge_attr': edge_attr,
            'site_mask': torch.tensor(sites, dtype=torch.bool)}

def collate(graphs):
    """Batch several graphs into one disconnected graph (what DGL's batching does)."""
    xs, eis, eas, masks, batch = [], [], [], [], []
    offset = 0
    for gi, g in enumerate(graphs):
        n = g['x'].shape[0]
        xs.append(g['x']); eas.append(g['edge_attr']); masks.append(g['site_mask'])
        eis.append(g['edge_index'] + offset)
        batch.append(torch.full((n,), gi, dtype=torch.long))
        offset += n
    return {'x': torch.cat(xs), 'edge_index': torch.cat(eis, dim=1), 'edge_attr': torch.cat(eas),
            'site_mask': torch.cat(masks), 'batch': torch.cat(batch), 'n_graphs': len(graphs)}

In [7]:
def build_set(kind):
    sub = train_raw[train_raw['marvin_pKa_type'] == kind]
    graphs, y, smiles, dropped = [], [], [], 0
    for mol, pka in zip(sub['ROMol'], sub['pKa']):
        if mol is None:
            dropped += 1; continue
        neu = neutralize(mol)
        g = mol_to_graph(neu, kind)
        if g['site_mask'].sum() == 0:
            dropped += 1; continue
        graphs.append(g); y.append(pka); smiles.append(Chem.MolToSmiles(neu))
    print(f'{kind}: kept {len(graphs)}, dropped {dropped} (no site / unparseable), '
          f'pKa range {min(y):.1f} to {max(y):.1f}')
    return graphs, torch.tensor(y, dtype=torch.float32), smiles

data = {k: build_set(k) for k in ['acidic', 'basic']}

acidic: kept 2558, dropped 96 (no site / unparseable), pKa range 2.0 to 12.0


basic: kept 3255, dropped 7 (no site / unparseable), pKa range 2.0 to 11.9


## 5. The Graph-pKa model (multi-instance pooling, Eq. 3 and 6, in log-sum-exp form)

In [8]:
def scatter_sum(src, index, n):
    out = torch.zeros((n,) + src.shape[1:], device=src.device, dtype=src.dtype)
    return out.index_add_(0, index, src)

def edge_softmax(logits, dst, n_nodes):
    """Softmax of edge logits over all edges arriving at the same destination node."""
    logits = logits.squeeze(-1)
    mx = torch.full((n_nodes,), -1e30, device=logits.device).scatter_reduce(
        0, dst, logits, reduce='amax', include_self=True)
    e = torch.exp(logits - mx.detach()[dst])   # shift is a constant: value & gradient unchanged
    denom = scatter_sum(e, dst, n_nodes)
    return (e / (denom[dst] + 1e-16)).unsqueeze(-1)

class GetContext(nn.Module):
    def __init__(self, node_in, edge_in, G, dropout):
        super().__init__()
        self.project_node  = nn.Sequential(nn.Linear(node_in, G), nn.LeakyReLU())
        self.project_edge1 = nn.Sequential(nn.Linear(node_in + edge_in, G), nn.LeakyReLU())
        self.project_edge2 = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * G, 1), nn.LeakyReLU())
        self.edge_transform = nn.Sequential(nn.Dropout(dropout), nn.Linear(G, G))
        self.gru = nn.GRUCell(G, G)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        n = x.shape[0]
        hv_new = self.project_node(x)
        he1 = self.project_edge1(torch.cat([x[src], edge_attr], dim=1))
        logits = self.project_edge2(torch.cat([hv_new[dst], he1], dim=1))
        a = edge_softmax(logits, dst, n)
        c = scatter_sum(a * self.edge_transform(he1), dst, n)
        return F.relu(self.gru(F.elu(c), hv_new))

class GNNLayer(nn.Module):
    def __init__(self, G, dropout):
        super().__init__()
        self.project_edge = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * G, 1), nn.LeakyReLU())
        self.project_node = nn.Sequential(nn.Dropout(dropout), nn.Linear(G, G))
        self.gru = nn.GRUCell(G, G)

    def forward(self, h, edge_index):
        src, dst = edge_index
        n = h.shape[0]
        logits = self.project_edge(torch.cat([h[dst], h[src]], dim=1))
        a = edge_softmax(logits, dst, n)
        c = scatter_sum(a * self.project_node(h)[src], dst, n)
        return F.relu(self.gru(F.elu(c), h))

class GraphPKa(nn.Module):
    def __init__(self, kind, node_in, edge_in, G=200, num_layers=6, dropout=0.1):
        super().__init__()
        assert kind in ('acidic', 'basic')
        self.kind = kind
        self.init_context = GetContext(node_in, edge_in, G, dropout)
        self.gnn_layers = nn.ModuleList([GNNLayer(G, dropout) for _ in range(num_layers - 1)])
        self.predict = nn.Sequential(nn.Dropout(dropout), nn.Linear(G, 1))

    def forward(self, b, return_atoms=False):
        h = self.init_context(b['x'], b['edge_index'], b['edge_attr'])
        for layer in self.gnn_layers:
            h = layer(h, b['edge_index'])
        atom_pka = self.predict(h).squeeze(-1)                     # Eq. 1 / 4
        ln10 = math.log(10.0)
        # Eq. 3 / 6 in log-sum-exp form; masked (non-site) atoms get -inf weight
        z = (-atom_pka if self.kind == 'acidic' else atom_pka) * ln10
        z = z.masked_fill(~b['site_mask'], float('-inf'))
        n_g, batch = b['n_graphs'], b['batch']
        mx = torch.full((n_g,), float('-inf'), device=z.device).scatter_reduce(
            0, batch, z, reduce='amax', include_self=True)
        mx = mx.detach()                           # log-sum-exp shift; value & gradient unchanged
        s = scatter_sum(torch.exp(z - mx[batch]), batch, n_g)
        lse = mx + torch.log(s)
        mol_pka = (-lse if self.kind == 'acidic' else lse) / ln10
        if return_atoms:
            return mol_pka, atom_pka
        return mol_pka

print('Model defined.')

Model defined.


## 6. Train the acidic and basic models (internal test = held-out 15%)

In [9]:
def to_device(b):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}

def batches(idx, graphs, y, bs, shuffle):
    idx = list(idx)
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        chunk = [int(j) for j in idx[i:i + bs]]
        yield to_device(collate([graphs[j] for j in chunk])), y[chunk].to(DEVICE)

@torch.no_grad()
def predict(model, graphs, idx, bs=128):
    model.eval()
    out = []
    for b, _ in batches(idx, graphs, torch.zeros(len(graphs)), bs, False):
        out.append(model(b).cpu())
    return torch.cat(out) if out else torch.tensor([])

def metrics(y_true, y_pred):
    err = (y_pred - y_true)
    mae = err.abs().mean().item()
    rmse = err.pow(2).mean().sqrt().item()
    r2 = 1 - err.pow(2).sum().item() / (y_true - y_true.mean()).pow(2).sum().item()
    return mae, rmse, r2

def train_model(kind, max_epochs=MAX_EPOCHS, patience=PATIENCE, bs=64, lr=1e-3):
    graphs, y, _ = data[kind]
    n = len(graphs)
    perm = np.random.RandomState(SEED).permutation(n)
    tr, va, te = perm[:int(.70*n)], perm[int(.70*n):int(.85*n)], perm[int(.85*n):]
    model = GraphPKa(kind, N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best, best_state, wait = float('inf'), None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for b, yt in batches(tr, graphs, y, bs, True):
            opt.zero_grad()
            loss = F.mse_loss(model(b), yt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        val_mae = metrics(y[va], predict(model, graphs, va))[0]
        if val_mae < best - 1e-4:
            best, best_state, wait = val_mae, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if ep % 5 == 0 or ep == 1:
            print(f'[{kind}] epoch {ep:3d}  val MAE {val_mae:.3f}  (best {best:.3f})')
        if wait >= patience:
            print(f'[{kind}] early stop at epoch {ep}')
            break
    model.load_state_dict(best_state)
    mae, rmse, r2 = metrics(y[te], predict(model, graphs, te))
    print(f'[{kind}] TEST  MAE {mae:.3f} | RMSE {rmse:.3f} | R2 {r2:.3f}')
    return model, {'train_idx': tr, 'test_idx': te, 'test_mae': mae, 'test_rmse': rmse, 'test_r2': r2, 'epochs': ep}

In [10]:
models, info = {}, {}
for kind in ["acidic", "basic"]:
    models[kind], info[kind] = train_model(kind)
    torch.save(models[kind].state_dict(), f"graphpka_{kind}.pt")

# internal test, both models together
yt, yp = [], []
for kind in ["acidic", "basic"]:
    g, y, _ = data[kind]; te = info[kind]["test_idx"]
    yt.append(y[te]); yp.append(predict(models[kind], g, te))
yt, yp = torch.cat(yt), torch.cat(yp)
METRICS = []
for kind in ["acidic", "basic"]:
    METRICS.append(dict(model="Graph-pKa (GNN)", test_set=f"internal 15% ({kind})", n=len(info[kind]["test_idx"]),
                        MAE=info[kind]["test_mae"], RMSE=info[kind]["test_rmse"], R2=info[kind]["test_r2"]))
mae, rmse, r2 = metrics(yt, yp)
METRICS.append(dict(model="Graph-pKa (GNN)", test_set="internal 15% (both)", n=len(yt), MAE=mae, RMSE=rmse, R2=r2))
pd.DataFrame(METRICS).round(3)

[acidic] epoch   1  val MAE 2.160  (best 2.160)


[acidic] epoch   5  val MAE 2.169  (best 2.152)


[acidic] epoch  10  val MAE 2.147  (best 2.147)


[acidic] epoch  15  val MAE 2.156  (best 2.147)


[acidic] epoch  20  val MAE 2.019  (best 2.019)


[acidic] epoch  25  val MAE 1.211  (best 1.211)


... (55 outputs left out)

[basic] epoch 145  val MAE 0.612  (best 0.587)


[basic] epoch 150  val MAE 0.580  (best 0.568)


[basic] TEST  MAE 0.553 | RMSE 0.867 | R2 0.851


,model,test_set,n,MAE,RMSE,R2
0,Graph-pKa (GNN),internal 15% (acidic),384,0.812,1.179,0.780
1,Graph-pKa (GNN),internal 15% (basic),489,0.553,0.867,0.851
2,Graph-pKa (GNN),internal 15% (both),873,0.667,1.016,0.823


## 7. External test sets (never seen in training)

In [11]:
def ext_eval(e):
    yt, yp, skipped = [], [], 0
    for kind in ["acidic", "basic"]:
        sub = e[e.marvin_pKa_type == kind]
        gs, ys = [], []
        for mol, p in zip(sub.ROMol, sub.pKa):
            if mol is None: skipped += 1; continue
            g = mol_to_graph(neutralize(mol), kind)
            if g["site_mask"].sum() == 0: skipped += 1; continue
            gs.append(g); ys.append(p)
        if gs:
            yt.append(torch.tensor(ys, dtype=torch.float32)); yp.append(predict(models[kind], gs, range(len(gs))))
    yt, yp = torch.cat(yt), torch.cat(yp)
    return len(yt), skipped, metrics(yt, yp)

for name, e in ext.items():
    n, sk, (mae, rmse, r2) = ext_eval(e)
    METRICS.append(dict(model="Graph-pKa (GNN)", test_set=name, n=n, MAE=mae, RMSE=rmse, R2=r2))
    print(f"{name}: n={n} (skipped {sk})  MAE {mae:.3f}  RMSE {rmse:.3f}  R2 {r2:.3f}")

Novartis: n=280 (skipped 0)  MAE 1.173  RMSE 1.561  R2 0.541
AvLiLuMoVe: n=119 (skipped 4)  MAE 0.573  RMSE 0.886  R2 0.848


## 8. Our ions: load, and the ions with a measured pKa

In [12]:
NAME = "IL_ion_dataset_579_v5_pKa_fixed.csv"
if "google.colab" in sys.modules:
    from google.colab import files
    up = files.upload(); NAME = list(up.keys())[0]
    ions = pd.read_csv(NAME)
else:
    ions = pd.read_csv(os.environ.get("ION_CSV", NAME))
print(len(ions), "ions")

579 ions


In [13]:
@torch.no_grad()
def mc_dropout_predict(model, graph, n_passes=30):
    model.train()                          # keeps dropout active on purpose
    b = to_device(collate([graph] * n_passes))
    preds = model(b).cpu().numpy()
    model.eval()
    return float(preds.mean()), float(preds.std())

def fp(mol):
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)

train_fps = {}
for kind in ['acidic', 'basic']:
    _, _, smis = data[kind]
    tr_idx = info[kind]['train_idx']
    train_fps[kind] = [fp(Chem.MolFromSmiles(smis[i])) for i in tr_idx]

def max_similarity(mol, kind):
    return max(DataStructs.BulkTanimotoSimilarity(fp(mol), train_fps[kind]))
print('Uncertainty and applicability-domain tools ready.')

Uncertainty and applicability-domain tools ready.


In [14]:
TRAIN_RANGE = (2.0, 12.0)
rows = []
for _, r in ions.iterrows():
    out = {"ion_id": r["ion_id"], "name": r["name"], "ion_type": r["ion_type"], "smiles": r["smiles"],
           "pKa_status": r["pKa_status"], "pKa": r["pKa"], "pKa_confidence": r["pKa_confidence"]}
    if r["pKa_status"] == "permanent cation (no pKa)":
        out["gnn_outcome"] = "permanent cation: no pKa"; rows.append(out); continue
    mol = Chem.MolFromSmiles(str(r["smiles"]))
    if mol is None:
        out["gnn_outcome"] = "unparseable SMILES"; rows.append(out); continue
    neu = neutralize(mol)
    kind = "acidic" if r["ion_type"] == "anion" else "basic"
    if Chem.GetFormalCharge(neu) != 0:
        out["gnn_outcome"] = "not predictable: cannot be neutralized"; rows.append(out); continue
    g = mol_to_graph(neu, kind)
    if g["site_mask"].sum() == 0:
        out["gnn_outcome"] = f"not predictable: no {kind} site (paper definition)"; rows.append(out); continue
    mean, std = mc_dropout_predict(models[kind], g)
    sim = max_similarity(neu, kind)
    out.update({"gnn_model": kind, "gnn_pKa": round(mean, 2), "gnn_mc_std": round(std, 2),
                "gnn_max_train_similarity": round(sim, 2), "gnn_in_domain": sim > 0.5, "gnn_outcome": "predicted"})
    rows.append(out)
pred = pd.DataFrame(rows)
print(pred.gnn_outcome.value_counts().to_string())

gnn_outcome
predicted                                             347
permanent cation: no pKa                              220
not predictable: cannot be neutralized                 10
not predictable: no acidic site (paper definition)      2


In [15]:
conf = pred.pKa_confidence.fillna("").str.lower()
known = pred[((pred.pKa_status == "measured") | ((pred.pKa_status == "web lookup") & conf.str.startswith("measured |")))].copy()
ok = known[known.gnn_outcome == "predicted"].copy()
ok["abs_err"] = (ok.pKa - ok.gnn_pKa).abs()
def m3(df):
    t, p = torch.tensor(df.pKa.values, dtype=torch.float32), torch.tensor(df.gnn_pKa.values, dtype=torch.float32)
    return metrics(t, p)
inr = ok[ok.pKa.between(*TRAIN_RANGE)]; outr = ok[~ok.pKa.between(*TRAIN_RANGE)]
for label, df in [("our measured ions (all)", ok), ("our measured ions (true pKa 2-12)", inr), ("our measured ions (true pKa outside 2-12)", outr)]:
    mae, rmse, r2 = m3(df)
    METRICS.append(dict(model="Graph-pKa (GNN)", test_set=label, n=len(df), MAE=mae, RMSE=rmse, R2=r2))
print(f"measured ions: {len(known)}   predicted: {len(ok)}")
print(pd.DataFrame(METRICS).round(3).to_string(index=False))
print("\nLargest errors:")
print(ok.sort_values("abs_err", ascending=False).head(10)[["ion_id","name","pKa","gnn_pKa","abs_err","gnn_in_domain"]].to_string(index=False))

measured ions: 153   predicted: 153
          model                                  test_set   n   MAE  RMSE     R2
Graph-pKa (GNN)                     internal 15% (acidic) 384 0.812 1.179  0.780
Graph-pKa (GNN)                      internal 15% (basic) 489 0.553 0.867  0.851
Graph-pKa (GNN)                       internal 15% (both) 873 0.667 1.016  0.823
Graph-pKa (GNN)                                  Novartis 280 1.173 1.561  0.541
Graph-pKa (GNN)                                AvLiLuMoVe 119 0.573 0.886  0.848
Graph-pKa (GNN)                   our measured ions (all) 153 2.075 3.439  0.134
Graph-pKa (GNN)         our measured ions (true pKa 2-12) 135 1.463 2.436  0.246
Graph-pKa (GNN) our measured ions (true pKa outside 2-12)  18 6.667 7.485 -0.978

Largest errors:
 ion_id                name   pKa  gnn_pKa  abs_err gnn_in_domain
 AN_001             bromide -9.00     4.23    13.23         False
 AN_111            chloride -6.30     4.15    10.45         False
 AN_153    diethylph

## 9. The ions that still had no pKa, and save

In [16]:
targets = ["AN_N001", "AN_N002", "CAT_N021", "CAT_N036", "AN_N028", "AN_N045"]
print(pred[pred.ion_id.isin(targets)][["ion_id","name","gnn_pKa","gnn_mc_std","gnn_in_domain","gnn_outcome"]].to_string(index=False))
pred.to_csv("IL_ion_pKa_GraphPKa_GNN.csv", index=False)
pd.DataFrame(METRICS).round(3).to_csv("GraphPKa_GNN_metrics.csv", index=False)
if "google.colab" in sys.modules:
    files.download("IL_ion_pKa_GraphPKa_GNN.csv"); files.download("GraphPKa_GNN_metrics.csv")
print("saved")

  ion_id                                   name  gnn_pKa  gnn_mc_std gnn_in_domain                            gnn_outcome
 AN_N001                           acesulfamate     6.79        0.59         False                              predicted
 AN_N002                               geranate     4.43        0.19         False                              predicted
CAT_N021                  didecylmethylammonium    10.14        0.38         False                              predicted
CAT_N036 clofaziminium (protonated clofazimine)     6.93        0.46         False                              predicted
 AN_N028                     5-nitrobarbiturate     7.20        0.35         False                              predicted
 AN_N045                      tetraphenylborate      NaN         NaN           NaN not predictable: cannot be neutralized
saved
